<a href="https://colab.research.google.com/github/DrKenReid/CNN-Tutorial---X-ray-image-classifier/blob/main/CNN_Tutorial_X_ray_image_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Convolutional Neural Networks (CNNs) for X-ray Pneumonia Detection

## Understanding CNNs

Convolutional Neural Networks (CNNs) are a class of deep learning models that have revolutionized the field of computer vision. Inspired by the human visual cortex, CNNs are designed to automatically and adaptively learn spatial hierarchies of features from input images.

### Why CNNs Excel in Image Processing

CNNs are particularly well-suited for image analysis tasks for several reasons:

1. **Local Connectivity**: CNNs use local receptive fields, allowing them to capture spatial relationships in images.
2. **Parameter Sharing**: Convolutional layers use the same set of weights across the entire image, significantly reducing the number of parameters.
3. **Spatial Hierarchy**: Through multiple layers, CNNs can learn to recognize complex patterns by combining simpler features.
4. **Translation Invariance**: CNNs can detect features regardless of their position in the image.

### Applications in Medical Imaging

In medical imaging, CNNs have shown remarkable success in various tasks, including:

- Disease detection and classification
- Organ and tumor segmentation
- Medical image reconstruction

For our specific case of pneumonia detection from chest X-rays, CNNs can learn to identify subtle patterns and anomalies that may be indicative of the disease.

## Guide Overview

In this tutorial, we'll walk through the process of building, training, and evaluating a CNN for pneumonia detection using chest X-ray images. Here's an overview of the steps we'll cover:

1. **Data Preparation**:
   - Downloading and extracting the pneumonia X-ray dataset
   - Organizing the data into training, validation, and test sets

2. **Data Preprocessing**:
   - Resizing images to a standard size
   - Normalizing pixel values
   - Applying data augmentation techniques

3. **Model Architecture**:
   - Designing a CNN structure suitable for X-ray image analysis
   - Implementing the model using TensorFlow and Keras

4. **Model Training**:
   - Setting up the training process with appropriate hyperparameters
   - Implementing callbacks for early stopping and learning rate adjustment
   - Training the model on the prepared dataset

5. **Model Evaluation**:
   - Assessing the model's performance on the test set
   - Generating and interpreting the confusion matrix
   - Analyzing precision, recall, and F1-score

6. **Visualization and Interpretation**:
   - Visualizing sample predictions alongside actual labels
   - Interpreting the model's performance in a medical context

7. **Model Saving and Loading**:
   - Saving the trained model and its history
   - Loading the model for future use or deployment

By following this guide, medical professionals will gain hands-on experience in applying deep learning techniques to a real-world medical imaging problem.

Thanks to [towardsdatascience for the original tutorial](https://towardsdatascience.com/medical-x-ray-%EF%B8%8F-image-classification-using-convolutional-neural-network-9a6d33b1c2a). Much of the documentation and updates to modern methods and removal of deprecated functions by Ken Reid.

# Connect to Your Google Drive

Google Colab (the IDE we're using - the thing you're looking at!) only stores data temporarily while you're using it. You can download results and such manually to your computer, or just store it to your Google Drive.

In [ ]:
import os

# This cell is Colab-specific: fall back gracefully when running elsewhere
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Mount Google Drive if not already mounted
    if not os.path.exists('/content/drive'):
        print("Mounting Google Drive...")
        drive.mount('/content/drive')
    else:
        print("Google Drive is already mounted.")
else:
    print("Not running in Google Colab - skipping Drive mount.")
    print("Note: model/history save paths later in the notebook point to Google Drive;")
    print("adjust them to local paths if running outside Colab.")

# Kaggle API

Here we automate the process of installing the Kaggle library, securely managing the user's Kaggle credentials, and verifying the API setup. Kaggle allows easy access to download large X-ray datasets, for training and testing our CNN model for pneumonia detection.

To use Kaggle, you must first create an account, then create a token under the "API Tokens" section of the [Settings](https://www.kaggle.com/settings/account) page. You can create several tokens and revoke them individually. Tokens require `kaggle` CLI version 1.8.0 or newer, which the cell below installs.

Creating a token downloads a `kaggle.json` file. When running the cell below, upload that file when prompted.

In [ ]:
import os

# The notebook is Colab-first; fall back gracefully when running locally
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def setup_kaggle_api():
    print("Setting up Kaggle API for X-ray dataset access...")

    # Install Kaggle library
    # The '!' allows us to run shell commands in the notebook
    # kaggle >= 1.8.0 is required for Kaggle's new API tokens
    # (-U upgrades the older version preinstalled in Colab)
    !pip install -q -U "kaggle>=1.8.0"

    kaggle_json = os.path.expanduser('~/.kaggle/kaggle.json')

    # Check if kaggle.json already exists
    if not os.path.exists(kaggle_json):
        if not IN_COLAB:
            raise FileNotFoundError(
                "No Kaggle credentials found. Download kaggle.json from "
                f"https://www.kaggle.com/settings/account and place it at {kaggle_json}"
            )
        print("\nPlease upload your kaggle.json file.")
        print("You can find this file in your Kaggle account settings.")
        uploaded = files.upload()

        if 'kaggle.json' not in uploaded:
            raise ValueError("kaggle.json was not uploaded. Please try again.")

        # Create .kaggle folder and move the file
        # These are shell commands to manage files in the notebook environment
        !mkdir -p ~/.kaggle
        !mv kaggle.json ~/.kaggle/

        # Set correct permissions
        # This ensures your Kaggle key is secure
        !chmod 600 ~/.kaggle/kaggle.json

    # Verify the setup.
    # Note: a failing '!' shell command does NOT raise a Python exception,
    # so we use subprocess and check the return code explicitly.
    import subprocess
    result = subprocess.run(
        ['kaggle', 'datasets', 'list', '-s', 'x-ray'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("\nKaggle API setup successful!")
        print("You can now access X-ray datasets from Kaggle.")
    else:
        print("\nError verifying Kaggle API setup:")
        print(result.stderr)
        print("Please check your kaggle.json file and try again.")

# Run the setup
setup_kaggle_api()

Here we acquire and prepare the pneumonia X-ray dataset for analysis. It first checks if the dataset is already available in the specified directory (i.e., if you're running this code multiple times, you don't need to download the data repeatedly).

If not, it proceeds to download the dataset from Kaggle, and then extracts the contents of the zip file.

In [ ]:
import os
import zipfile

# Define file paths
zip_file = "/content/pneumonia-xray-images.zip"
target_dir = "/content/dataset/cnn/pneumonia_revamped"

# Check if the dataset is already extracted
if os.path.exists(target_dir) and os.listdir(target_dir):
    print(f"Dataset already exists in {target_dir}")
else:
    # Check if the zip file is already downloaded
    if not os.path.exists(zip_file):
        print("Downloading the dataset...")
        !kaggle datasets download -d pcbreviglieri/pneumonia-xray-images
    else:
        print("Zip file already downloaded.")

    # Extract the contents
    print(f"Extracting dataset to {target_dir}")
    with zipfile.ZipFile(zip_file, 'r') as zfile:
        zfile.extractall(target_dir)

    print(f"Dataset extracted to {target_dir}")

print("\nDataset is ready for use.")

# 1 Set up

These library imports provide essential tools for data scientists working on image analysis projects:

1. **Matplotlib** enables the creation of various visualizations, e.g., for displaying X-ray images and plotting analysis results.

2. **NumPy** facilitates efficient numerical operations and array manipulations, which are fundamental for image processing and data analysis tasks.

3. **Pandas** offers powerful data manipulation capabilities, useful for handling metadata associated with the images and organizing analysis results in a structured format.

4. **Random seeds** are set for TensorFlow and NumPy to ensure reproducibility — running the notebook again should produce similar (though not identical on GPU) results.

In [ ]:
# Matplotlib: For creating visualizations (e.g., displaying X-ray images, plotting results)
import matplotlib.pyplot as plt

# NumPy: For numerical operations and working with arrays (e.g., image processing)
import numpy as np

# Pandas: For data manipulation and analysis (e.g., handling metadata, organizing results)
import pandas as pd

# Set random seeds for reproducibility
# This ensures results are consistent across runs (as much as possible with GPU non-determinism)
import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)

## 1.1 Define Constants

Then we store the directories of where things are stored.

In [ ]:
# Path to the training dataset
train_path = '/content/dataset/cnn/pneumonia_revamped/train'

# Path to the testing dataset
test_path = '/content/dataset/cnn/pneumonia_revamped/test'

# Path to the validation dataset
valid_path = '/content/dataset/cnn/pneumonia_revamped/val'

Researchers might modify the below parameters for several reasons:

**`batch_size`**: Adjusting this impacts training speed and memory usage. Larger batches can lead to faster training but require more GPU memory, while smaller batches might improve generalization.

**`img_height` & `img_width`**: Changing dimensions affects the input size to the CNN. Larger dimensions retain more image detail but increase computational requirements. 150×150 is a good balance between speed and detail for this tutorial — training completes in minutes rather than hours. If you want higher fidelity, try 256×256 or even 512×512, but note that training time and memory usage increase quadratically with image size.

In [ ]:
#Define standard parameter values
batch_size = 32  # Number of images processed in each training iteration
img_height = 150  # Height of input images in pixels
img_width = 150   # Width of input images in pixels

# 2 Preparing Data

## 2.1 Image Data Augmentation

Here we:
*   Define an augmentation function to enhance dataset diversity
*   Load training, testing, and validation datasets from directories
*   Apply data augmentation to the **training set only**
*   Normalize pixel values for all datasets (0-255 to 0-1 range)
*   Set image size, batch size, and color mode (grayscale) for consistency
*   Use binary labels for classification
*   Implement prefetching to optimize data loading performance
*   Print the number of batches in each dataset for verification

In [ ]:
# Function to apply data augmentation to training images
def augment(image, label):
    # Randomly flip the image horizontally
    image = tf.image.random_flip_left_right(image)

    # Randomly adjust brightness (±20%)
    image = tf.image.random_brightness(image, max_delta=0.2)

    # Randomly adjust contrast
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)

    # Clip pixel values to valid [0, 1] range after augmentation
    image = tf.clip_by_value(image, 0.0, 1.0)

    return image, label

# Load and prepare the training dataset
train = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    color_mode='grayscale',
    label_mode='binary'
)
# Cache → normalize → augment → shuffle → prefetch (optimal pipeline order)
train = train.cache()
train = train.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y))
train = train.map(augment)
train = train.shuffle(buffer_size=1000, seed=42)
train = train.prefetch(buffer_size=tf.data.AUTOTUNE)

# Load and prepare the testing dataset (NO augmentation, NO shuffle)
test = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    color_mode='grayscale',
    label_mode='binary',
    shuffle=False
)
test = test.cache()
test = test.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y))
test = test.prefetch(buffer_size=tf.data.AUTOTUNE)

# Load and prepare the validation dataset (NO augmentation)
valid = tf.keras.utils.image_dataset_from_directory(
    valid_path,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    color_mode='grayscale',
    label_mode='binary'
)
valid = valid.cache()
valid = valid.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y))
valid = valid.prefetch(buffer_size=tf.data.AUTOTUNE)

# Print dataset sizes (number of batches)
print(f"Training batches:   {tf.data.experimental.cardinality(train).numpy()}")
print(f"Test batches:       {tf.data.experimental.cardinality(test).numpy()}")
print(f"Validation batches: {tf.data.experimental.cardinality(valid).numpy()}")

This code visualizes a sample of images from the training dataset:

*    Takes a single batch from the training dataset using `.take(1)`
*    Displays up to 10 images in a 2×5 grid
*    Labels each image with its class name ('NORMAL' or 'PNEUMONIA')
*    Uses grayscale colormap since the images are single-channel X-rays

This visualization helps verify that:
- The dataset loaded correctly
- Images are properly normalized
- Labels match the visual content

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create a large figure (20 inches wide, 12 inches tall)
plt.figure(figsize=(20, 12))

# Dictionary to map class indices to labels
dic = {0: 'NORMAL', 1: 'PNEUMONIA'}

# Take one batch from the training dataset
for X_batch, Y_batch in train.take(1):
    # Display up to 10 images from this batch
    num_images = min(10, X_batch.shape[0])
    for i in range(num_images):
        # Create a subplot in a 2x5 grid
        plt.subplot(2, 5, i + 1)

        # Select the i-th image and label
        image = X_batch[i]
        label = int(Y_batch[i].numpy().item())

        # Set the title of the subplot to the class of the image
        plt.title(dic[label], fontsize=14)

        # Turn off axis labels
        plt.axis('off')

        # Display the image
        # np.squeeze removes the single channel dimension for grayscale display
        plt.imshow(np.squeeze(image), cmap='gray', interpolation='nearest')

# Adjust the layout to prevent overlap
plt.tight_layout()

# Display the plot
plt.show()

## 2.2 Data Augmentation Explained

Data augmentation artificially increases the variety in the training data, helping the model generalize better to new, unseen X-rays.

**Why we augment:**
- **Robustness**: The model becomes more robust to variations it might encounter in real-world scenarios, like differences in X-ray brightness or slight rotations.
- **Overfitting Prevention**: By introducing random variations, we help prevent the model from memorizing the exact training images.
- **Balanced Approach**: The parameters are set to introduce variability without distorting the images so much that they become unrealistic or lose important features.

**What the active pipeline (Section 2.1) applies:**
- **Horizontal flip**: Used cautiously — while real X-rays can be flipped, the heart is normally on the left side, so some practitioners avoid this augmentation to preserve anatomical orientation.
- **Brightness and contrast jitter (±20%)**: Mimics differences in exposure and acquisition settings between X-ray machines.
- **Vertical flip**: Not used, as upside-down X-rays are unrealistic.

The legacy example below additionally shows small rotations (±15°), zoom, and shifts — reasonable augmentations for chest X-rays, but **not** part of the pipeline this tutorial actually trains with.

> **Note:** In Section 2.1 above, we applied augmentation using `tf.image` functions within the `tf.data` pipeline — the modern, recommended approach. The cell below shows an **alternative legacy approach** using `ImageDataGenerator` for educational purposes. `ImageDataGenerator` is deprecated in modern TensorFlow and should not be used in new projects.

In [ ]:
# ⚠️ LEGACY APPROACH — shown for educational purposes only.
# The modern tf.data augmentation pipeline (Section 2.1) is what this tutorial actually uses.
# ImageDataGenerator is deprecated in TensorFlow 2.9+ and will be removed in future versions.

from tensorflow.keras.preprocessing.image import ImageDataGenerator

image_gen = ImageDataGenerator(
    rotation_range=15,          # Randomly rotate images by up to 15 degrees
    zoom_range=0.2,             # Randomly zoom image in or out by up to 20%
    width_shift_range=0.1,      # Randomly shift images horizontally by up to 10%
    height_shift_range=0.1,     # Randomly shift images vertically by up to 10%
    horizontal_flip=True,       # Randomly flip images horizontally
    vertical_flip=False,        # Don't flip vertically (unrealistic for chest X-rays)
    shear_range=0.2,            # Randomly apply shearing transformations
    brightness_range=(0.8, 1.2) # Randomly adjust brightness ±20%
)

# This ImageDataGenerator is NOT used in training — it is here only to show
# what the legacy API looks like. See Section 2.1 for the active augmentation code.

# 3 Tensorflow - Keras

These imports provide essential tools for building and managing CNN models:

*    `Sequential`, `load_model`: For creating and loading models.
*    `Input`: Explicitly defines the input shape (modern Keras best practice).
*    `Dense`, `Conv2D`, `Flatten`, `MaxPooling2D`: Core layers for building CNN architectures.
*    `Dropout`: Randomly disables neurons during training to prevent overfitting.
*    `BatchNormalization`: Normalizes layer outputs to stabilize and accelerate training.
*    `EarlyStopping`, `ReduceLROnPlateau`: Callbacks to prevent overfitting and optimize training.

In [ ]:
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Conv2D, Flatten, MaxPooling2D, Dropout, Input, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

## 3.1 Convolutional Neural Network Model

CNN Structure Explanation:

1. **Input Layer**: Explicitly defines the expected input shape — grayscale X-ray images (150 × 150 × 1).

2. **Convolutional Blocks** (Conv2D → BatchNormalization → MaxPooling → Dropout):
   - 3 blocks with increasing filter counts (32 → 64 → 128)
   - Each MaxPooling halves the spatial dimensions: 150 → 74 → 36 → 17
   - The final feature map is 17×17×128 = 36,992 values — enough spatial information for the dense layers to classify from

3. **Flatten Layer**: Converts 2D feature maps to a 1D vector

4. **Dense Layers**:
   - Two hidden layers (128 and 64 units) with Dropout for regularization
   - Final layer (1 unit, sigmoid) for binary classification (pneumonia or not)

**Why 3 blocks instead of 5?** The number of pooling layers must match the image size. Each MaxPooling halves the dimensions, so with 150×150 images:

| Blocks | Final feature map | Total values | Issue |
|---|---|---|---|
| 3 | 17×17×128 | 36,992 | Good — plenty of spatial detail |
| 5 | 2×2×64 | 256 | Too compressed — model can't distinguish classes |

> **Rule of thumb:** Ensure your final feature map is at least 7×7 before flattening. If you increase `img_height`/`img_width`, you can add more conv blocks.

Key Design Decisions:

| Decision | Rationale |
|---|---|
| 32 → 64 → 128 filters | Captures simple features first, then complex ones |
| Dropout (0.25/0.5) | Regularizes the network — higher dropout in dense layers |
| BatchNormalization | Stabilizes and accelerates training |
| Adam(lr=0.0001) | Lower learning rate prevents the model from collapsing to one class |
| Sigmoid output | Outputs a probability for binary classification |

In [ ]:
# Initialize a sequential model (layers added in sequence)
cnn = Sequential()

# Explicit input layer (modern Keras best practice — avoids deprecation warnings)
cnn.add(Input(shape=(img_width, img_height, 1)))

# --- Convolutional Block 1 ---
# Input: 150×150×1 → Conv: 148×148×32 → Pool: 74×74×32
cnn.add(Conv2D(32, (3, 3), activation="relu"))
cnn.add(BatchNormalization())
cnn.add(MaxPooling2D(pool_size=(2, 2)))
cnn.add(Dropout(0.25))

# --- Convolutional Block 2 ---
# Input: 74×74×32 → Conv: 72×72×64 → Pool: 36×36×64
cnn.add(Conv2D(64, (3, 3), activation="relu"))
cnn.add(BatchNormalization())
cnn.add(MaxPooling2D(pool_size=(2, 2)))
cnn.add(Dropout(0.25))

# --- Convolutional Block 3 ---
# Input: 36×36×64 → Conv: 34×34×128 → Pool: 17×17×128
cnn.add(Conv2D(128, (3, 3), activation="relu"))
cnn.add(BatchNormalization())
cnn.add(MaxPooling2D(pool_size=(2, 2)))
cnn.add(Dropout(0.25))

# Flatten: Converts 2D feature maps (17×17×128 = 36,992) to 1D feature vector
cnn.add(Flatten())

# --- Dense (fully connected) layers for classification ---
cnn.add(Dense(128, activation='relu'))
cnn.add(Dropout(0.5))  # Higher dropout in dense layers to prevent overfitting

cnn.add(Dense(64, activation='relu'))
cnn.add(Dropout(0.5))

# Output layer: Sigmoid outputs a probability between 0 and 1
cnn.add(Dense(1, activation='sigmoid'))

# Compile the model with a lower learning rate for more stable training
cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',     # Standard loss for binary classification
    metrics=['accuracy']            # Track accuracy during training
)

In [ ]:
cnn.summary()

In [ ]:
# Import the plot_model utility from Keras
from tensorflow.keras.utils import plot_model

# Generate a plot of the CNN model
plot_model(
    cnn,                     # The model to visualize
    show_shapes=True,        # Show the shape of the inputs/outputs of each layer
    show_layer_names=True,   # Display the name of each layer
    rankdir='TB',            # 'TB' means top to bottom layout
    expand_nested=True       # Expand nested models/layers
)

## 3.2 Fit Model

This code sets up two important mechanisms to improve the training process of the CNN:

**Early Stopping (EarlyStopping)**:

Purpose:

*    Prevents overfitting by stopping the training process when the model stops improving.

How it works:

*    Monitors the validation loss ('`val_loss`')
*    Aims to minimize this loss ('`mode="min"`')
If there's no improvement for 3 consecutive epochs ('`patience=3`'), it stops training

Benefit:

*    Helps avoid overfitting, where the model might perform well on training data but poorly on new, unseen data.

**Learning Rate Reduction (ReduceLROnPlateau)**:

Purpose:

*    Adjusts the learning rate during training to fine-tune the model's performance.

How it works:

*    Monitors the validation loss if there's no improvement for 2 epochs ('`patience=2`'), it reduces the learning rate
*    The learning rate is reduced by multiplying it by 0.3 ('`factor=0.3`')
*    The learning rate will not go below 0.000001 ('`min_lr=0.000001`')

Benefit:

*    Helps the model converge to a better solution by making smaller adjustments as it gets closer to the optimum.

**Why these callbacks are important:**

1. Efficiency: Early stopping saves computational resources by preventing unnecessary training epochs.
2. Generalization: Both callbacks help in creating a model that generalizes well to new X-ray images, not just the training set.
3. Fine-tuning: Learning rate reduction allows the model to make finer adjustments as it gets closer to the optimal solution.
4. Automation: These callbacks automate part of the hyperparameter tuning process, reducing the need for manual intervention.
5. Preventing Overfitting: Especially crucial in medical applications where the model needs to perform well on diverse, real-world X-rays.

**For this specific pneumonia detection model:**

Early stopping will help ensure that the model doesn't start memorizing ONLY the training X-rays.
Learning rate reduction will help in fine-tuning the model's ability to detect subtle features in X-rays that indicate pneumonia.

In [ ]:
# Define Early Stopping callback
# patience=5 gives the model enough room to recover from temporary plateaus
early = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=5,
    restore_best_weights=True
)

# Define Learning Rate Reduction callback
learning_rate_reduction = ReduceLROnPlateau(
    monitor='val_loss',
    patience=3,          # Reduce LR after 3 epochs with no improvement
    verbose=1,
    factor=0.5,          # Halve the learning rate (gentler than 0.3)
    min_lr=0.000001
)

# Combine callbacks into a list
callbacks_list = [early, learning_rate_reduction]

**Label Extraction:**

The extract_labels function iterates through the train dataset.
It extracts the labels (y) from each batch of data.
These labels represent the classifications of the X-ray images (likely 0 for NORMAL and 1 for PNEUMONIA).


**Label Processing:**

The extracted labels are converted to integers to ensure compatibility with subsequent operations.
np.unique(train_labels) identifies the unique classes in the dataset (likely [0, 1]).


**Class Weight Computation:**

compute_class_weight calculates weights for each class based on their frequency in the dataset.
The 'balanced' strategy assigns higher weights to less frequent classes.
This helps address class imbalance, which is common in medical datasets where diseased cases might be less common than normal cases.


**Class Distribution:**

The code counts the number of samples for each class.
This provides insight into the balance (or imbalance) of the dataset.


**Output:**

The class weights are printed, showing the relative importance assigned to each class.
The class distribution is displayed, revealing the actual count of samples in each class.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Extract labels from the training dataset
def extract_labels(dataset):
    labels = []
    for _, y in dataset:
        labels.append(y.numpy())
    return np.concatenate(labels).flatten().astype(int)

train_labels = extract_labels(train)

# Get unique classes
classes = np.unique(train_labels)

# Compute class weights to handle imbalanced data
# 'balanced' assigns higher weights to under-represented classes
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_labels
)

# Keras class_weight requires plain Python int keys (not numpy ints)
cw = {int(cls): float(w) for cls, w in zip(classes, weights)}
print("Class weights:", cw)

# Print class distribution
print("\nClass distribution:")
unique, counts = np.unique(train_labels, return_counts=True)
for class_label, count in zip(unique, counts):
    label_name = 'NORMAL' if class_label == 0 else 'PNEUMONIA'
    print(f"  Class {class_label} ({label_name}): {count} samples")

### Training

**How training works:**

Each **epoch** is one complete pass over the entire training dataset. The model sees every image, computes predictions, measures the error (loss), and updates its weights to improve.

**Key points:**

- **Epochs = 25**: With a learning rate of 0.0001, the model needs more passes to converge. `EarlyStopping` (patience=5) will halt training automatically once it plateaus, so you won't necessarily run all 25.
- **Validation monitoring**: After each epoch, the model is evaluated on the validation set (data it hasn't trained on) to check for overfitting.
- **Class weights**: Since pneumonia and normal samples may not be equally represented, class weights ensure the model doesn't just predict the majority class.
- **Callbacks**: `EarlyStopping` prevents wasted computation, and `ReduceLROnPlateau` fine-tunes learning as the model converges.

**Important considerations:**

- **Computational resources**: Training CNNs on large image datasets is GPU-intensive. Colab provides free GPU access (Runtime → Change runtime type → GPU).
- **Model evaluation**: After training, we evaluate on a held-out test set for an unbiased performance estimate.
- **Iterative improvement**: Based on results, you may need to adjust the architecture, hyperparameters, or augmentation strategy and retrain.
- **Clinical validation**: Statistical performance on a test set is just the first step. Real-world deployment requires clinical validation and regulatory approval.

In [ ]:
# Train the CNN model
# epochs=25 with lr=0.0001 gives the model enough time to learn;
# EarlyStopping (patience=5) will halt training if validation loss plateaus
history = cnn.fit(
    train,
    epochs=25,
    validation_data=valid,
    class_weight=cw,
    callbacks=callbacks_list
)

## 3.3 Saving the Model

**Storage Space:** Deep learning models, especially those trained on image data, can be quite large. Ensure you have sufficient storage space.

**Security:** When dealing with models trained on medical data, ensure that your saved model doesn't inadvertently contain any sensitive patient information.

**Documentation:** It's good practice to document the performance metrics, dataset details, and training parameters along with your saved model.

**Compatibility:** Ensure you note the versions of TensorFlow and Keras used, as there might be compatibility issues when loading models with different software versions.

**Regulatory Compliance:** In a clinical setting, saving and versioning models properly is often required for regulatory compliance and audit trails.

In [ ]:
import json

# Define file paths (saved to Google Drive for persistence)
model_fp = "/content/drive/My Drive/cnn_pneu_vamp_model.keras"
history_fp = "/content/drive/My Drive/cnn_pneu_vamp_history.json"

# Custom JSON encoder to handle NumPy types
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

def save_model_and_history(model, history, model_fp, history_fp):
    """Save the trained model and training history to disk."""
    model.save(model_fp)
    with open(history_fp, 'w') as f:
        json.dump(history.history, f, cls=NumpyEncoder)
    print(f"Model saved to: {model_fp}")
    print(f"History saved to: {history_fp}")

def load_model_and_history(model_fp, history_fp):
    """Load a previously saved model and training history."""
    model = load_model(model_fp)  # Uses the load_model imported earlier
    with open(history_fp, 'r') as f:
        history = json.load(f)
    print("Model and history loaded successfully.")
    return model, history

# Save
save_model_and_history(cnn, history, model_fp, history_fp)

# Load (demonstrates how to restore the model later)
loaded_model, loaded_history = load_model_and_history(model_fp, history_fp)
print("Available metrics:", list(loaded_history.keys()))

## 3.4 Evaluation

**Test Set Importance:** The test set represents new, unseen data. Performance on this set is a better indicator of how well your model will generalize to new X-ray images in a clinical setting.

**Accuracy Interpretation:** The accuracy tells you what percentage of X-rays the model correctly classified (as pneumonia or normal). For example, an accuracy of 85% means the model correctly classified 85 out of 100 X-rays.

**Clinical Relevance:** While accuracy is important, in a medical context, you'll also want to consider other metrics like sensitivity (ability to correctly identify pneumonia cases) and specificity (ability to correctly identify normal cases).

**Benchmark Comparison:** This accuracy should be compared to both human-level performance and existing automated systems for pneumonia detection.


In a medical context, understanding the implications of false positives (unnecessarily worrying patients) and false negatives (missing cases of pneumonia) are critical, for obvious reasons.

In [ ]:
# Evaluate the model on the test dataset
test_accu = cnn.evaluate(test)

# Print the testing accuracy
print('The testing accuracy is:', test_accu[1]*100, '%')

Here we perform the evaluation of the trained CNN model on the test dataset. Here's a breakdown of its key components:

1. Label Extraction:
   - The `get_labels` function extracts the true labels from the test dataset.

2. Predictions:
   - `cnn.predict(test)` generates predictions for all samples in the test dataset.

3. Binary Classification:
   - Predictions (probabilities) are converted to binary class labels using a 0.5 threshold.

4. Performance Evaluation:
   - The `classification_report` provides a detailed breakdown of the model's performance, including precision, recall, and F1-score for each class.
   - The `confusion_matrix` shows the count of true positives, false positives, true negatives, and false negatives.

This evaluation process is crucial for assessing the model's performance in classifying X-ray images as normal or pneumonia. It provides insights into the model's accuracy, its ability to correctly identify positive cases (sensitivity), and its ability to correctly identify negative cases (specificity), which are all critical metrics in medical diagnostics.

### Understanding the Classification Metrics

**Classes:**
- `0` = NORMAL
- `1` = PNEUMONIA

| Metric | What it tells you |
|---|---|
| **Precision** | Of all images the model *predicted* as class X, what fraction were actually class X? High precision = few false positives. |
| **Recall** (Sensitivity) | Of all images that *actually are* class X, what fraction did the model correctly identify? High recall = few false negatives. |
| **F1-score** | Harmonic mean of precision and recall — a single number balancing both. |
| **Support** | Number of test samples in each class. |
| **Accuracy** | Overall fraction of correct predictions. |
| **Macro avg** | Simple average of metrics across classes (treats both classes equally). |
| **Weighted avg** | Average of metrics weighted by class support (accounts for class imbalance). |

**Clinical interpretation:**
- In medical screening, **high recall for PNEUMONIA** (class 1) is typically more important — a missed pneumonia case (false negative) is more dangerous than a false alarm (false positive).
- If recall for PNEUMONIA is high but precision is lower, the model is being *cautious* — it flags more cases as pneumonia to avoid missing any.
- The confusion matrix below gives you the raw counts to see exactly where errors occur.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Function to extract labels from a tf.data.Dataset
def get_labels(dataset):
    labels = []
    for _, y in dataset:
        labels.append(y.numpy())
    return np.concatenate(labels).flatten()

# Get true labels from the test set
true_labels = get_labels(test)

# Make predictions on the test dataset
predictions = cnn.predict(test)

# Convert predictions to binary class labels (0.5 threshold)
class_predictions = (predictions > 0.5).astype("int32").flatten()

# Generate and print the classification report
print("Classification Report:")
print(classification_report(true_labels, class_predictions,
                            target_names=["Normal", "Pneumonia"]))

# Generate and print the confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, class_predictions))

Below we visualize the training progress of a CNN model:

1. It creates two subplots: one for accuracy and one for loss.
2. Both training and validation metrics are plotted over epochs.
3. Final metrics are printed for quick reference.

Results interpretation:

1. Accuracy: Both training and validation accuracy improve rapidly and stabilize around 94%, indicating good learning and generalization.

2. Loss: Training and validation loss decrease quickly, with validation loss slightly lower, suggesting the model isn't overfitting.

3. Convergence: The model appears to converge after just 2-3 epochs, showing efficient learning on this dataset.

4. Performance: High final accuracies (>94%) suggest strong performance in distinguishing between normal and pneumonia X-rays.

5. Stability: The close alignment of training and validation curves indicates good model stability and generalization.

Overall, these results suggest a well-performing model that learns quickly and generalizes well to unseen data, which is promising for pneumonia detection applications.

In [ ]:
if history.history:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Plot accuracy
    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Validation')
    ax1.set_title('Model Accuracy')
    ax1.set_ylabel('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Plot loss
    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Validation')
    ax2.set_title('Model Loss')
    ax2.set_ylabel('Loss')
    ax2.set_xlabel('Epoch')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Print final metrics
    print(f"Final training accuracy:   {history.history['accuracy'][-1]:.4f}")
    print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")
    print(f"Final training loss:       {history.history['loss'][-1]:.4f}")
    print(f"Final validation loss:     {history.history['val_loss'][-1]:.4f}")
else:
    print("The history object is empty. Make sure the model has been trained.")

Below we perform final evaluation steps on the model's predictions:

1. Converts probability predictions to binary classifications (0 or 1) using a 0.5 threshold.

2. Prints the shape of the predictions array and the first few predictions for a quick check.

3. Extracts true labels from the test dataset.

4. Calculates and prints the overall accuracy of the model on the test set using sklearn's accuracy_score function.

This process provides a straightforward assessment of the model's performance, showing how well it classifies X-ray images into normal and pneumonia categories on unseen data. The accuracy score gives a single, interpretable metric of the model's overall effectiveness in this binary classification task.

In [ ]:
# Print prediction summary
print("Shape of predictions:", predictions.shape)
print("First 10 predictions (raw probabilities):", predictions[:10].flatten())
print("First 10 predictions (binary):", class_predictions[:10])

# Calculate and print accuracy
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(true_labels, class_predictions)
print(f"\nTest Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

Lets build a confusion matrix graphically:

In [ ]:
import seaborn as sns

# Reuse true_labels and class_predictions from the previous cell
cm = confusion_matrix(true_labels, class_predictions)

# Create a labeled DataFrame for clearer visualization
cm_df = pd.DataFrame(
    data=cm,
    index=["Actual Normal", "Actual Pneumonia"],
    columns=["Predicted Normal", "Predicted Pneumonia"]
)

# Plot heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

# Print classification report (with class names for readability)
print("\nClassification Report:")
print(classification_report(true_labels, class_predictions,
                            target_names=["Normal", "Pneumonia"]))

### ROC Curve and Choosing an Operating Threshold

So far we have converted probabilities to class labels with a fixed 0.5 threshold, but that choice is a *clinical decision*, not a mathematical constant. The ROC curve shows the trade-off between **sensitivity** (catching pneumonia cases) and **specificity** (not raising alarms on healthy patients) across *all* possible thresholds, and the area under it (**AUC**) summarizes how well the model separates the two classes independently of any single threshold.

For a screening tool you would typically choose a threshold favouring **high sensitivity** - missing a pneumonia case (false negative) is more dangerous than a false alarm - and accept the extra false positives that come with it. The table printed below makes that trade-off concrete.

In [ ]:
from sklearn.metrics import roc_curve, auc

# Compute the ROC curve from raw predicted probabilities (NOT thresholded labels)
fpr, tpr, roc_thresholds = roc_curve(true_labels, predictions.flatten())
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f'CNN (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random classifier (AUC = 0.5)')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC Curve - Pneumonia Detection')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Sensitivity/specificity trade-off at a few candidate operating thresholds
print(f"{'Threshold':>10} {'Sensitivity':>12} {'Specificity':>12}")
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    preds_t = (predictions.flatten() >= t).astype(int)
    tp = np.sum((preds_t == 1) & (true_labels == 1))
    fn = np.sum((preds_t == 0) & (true_labels == 1))
    tn = np.sum((preds_t == 0) & (true_labels == 0))
    fp = np.sum((preds_t == 1) & (true_labels == 0))
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    print(f"{t:>10.1f} {sensitivity:>12.3f} {specificity:>12.3f}")

### Why Validation and Test Results Can Disagree

Don't be surprised if this model posts high validation accuracy yet noticeably lower test accuracy. That gap is real, well documented for this benchmark, and arguably the most important lesson in this tutorial:

- **Image-level (not patient-level) splitting**: The underlying dataset (Kermany et al.) contains multiple X-rays from the same patients, and the train/validation split of this re-packaged version was made at the *image* level. Near-duplicate images of the same patient can therefore land on both sides of the split, letting the model "recognize" validation patients it effectively trained on. Rigorous medical ML always splits **by patient**, never by image.
- **Distribution shift**: The test set comes from the original benchmark's held-out collection and differs in acquisition characteristics from the training data, so it is a genuinely harder - and more honest - measure of generalization.
- **Optimistic early stopping**: We select the best epoch using validation loss, so the validation score is itself slightly optimistic by construction.

When you see a large validation-test gap, the model didn't suddenly "get worse" - the validation set was giving an inflated estimate all along. Diagnosing *why* a benchmark score won't survive contact with new data is exactly the skill that separates a benchmark demo from a clinically credible evaluation.

Below we extract all test images and labels into NumPy arrays, which makes it easy to:
- Index individual images by position
- Visualize specific predictions alongside their actual labels
- Perform custom analysis beyond what `tf.data` pipelines offer

The `extract_data` function iterates through the `tf.data.Dataset`, converting each batch from TensorFlow tensors to NumPy arrays, then concatenates them into a single array for images (`x`) and labels (`y`).

In [ ]:
# Extract all images and labels from the test dataset into NumPy arrays
# This makes it easier to index individual images for visualization
def extract_data(dataset):
    x_list, y_list = [], []
    for x_batch, y_batch in dataset:
        x_list.append(x_batch.numpy())
        y_list.append(y_batch.numpy())
    return np.concatenate(x_list), np.concatenate(y_list).flatten()

x, y = extract_data(test)

print("X shape:", x.shape)
print("Y shape:", y.shape)
print("X data type:", x.dtype)
print("Y data type:", y.dtype)
print("Unique values in Y:", np.unique(y))

print("\nClass distribution:")
for class_id, count in zip(*np.unique(y, return_counts=True)):
    print(f"  Class {int(class_id)} ({'NORMAL' if class_id == 0 else 'PNEUMONIA'}): {count} samples")

### Visual Inspection

Never just trust the numbers — let's visualize some predictions alongside the actual labels. Titles are colored **green** for correct predictions and **red** for incorrect ones.

In [ ]:
dic = {0: 'NORMAL', 1: 'PNEUMONIA'}
plt.figure(figsize=(20, 20))

# Show 9 sample predictions spread evenly across the test set.
# The test set is loaded without shuffling (NORMAL first, then PNEUMONIA),
# so an even spread covers both classes.
indices = np.linspace(0, len(y) - 1, 9, dtype=int)

for plot_idx, i in enumerate(indices):
    plt.subplot(3, 3, plot_idx + 1)

    prob_pneumonia = float(predictions[i][0])

    if prob_pneumonia >= 0.5:
        out = f'{prob_pneumonia:.2%} probability of being Pneumonia'
    else:
        out = f'{1 - prob_pneumonia:.2%} probability of being Normal'

    actual_label = dic.get(int(y[i]), 'Unknown')

    # Color the title green if correct, red if wrong
    predicted_class = 1 if prob_pneumonia >= 0.5 else 0
    is_correct = predicted_class == int(y[i])
    color = 'green' if is_correct else 'red'

    plt.title(f"{out}\nActual: {actual_label}", color=color, fontsize=11)
    plt.imshow(np.squeeze(x[i]), cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()

# Print overall accuracy using raw predictions (not already-thresholded values)
correct = np.sum((predictions.flatten() >= 0.5).astype(int) == y.astype(int))
print(f"Overall test accuracy: {correct / len(y):.2%}")